In [26]:
# =============================================================================
# FINE-TUNING WHISPER ON YOUR DOMAIN
# =============================================================================
#
# Course map:
#   Phase 04 → Transfer learning & fine-tuning (general)
#   Phase 06 → Whisper architecture & fine-tuning
#   Phase 11 → LoRA & QLoRA (train few extra weights, keep base frozen)
#
# Today's deliverable (target):
#   LoRA-tuned Whisper that hears SALES jargon better
#   (WER on terms like "SOC 2", "$299", "Professional" ~25% → <5%)
#   Drop-in replacement for the ASR step in your live earpiece loop.
#
# ---------------------------------------------------------------------------
# THE PROBLEM (easy picture)
# ---------------------------------------------------------------------------
# Stock Whisper is great on general English. On a sales call it still mishears:
#   "SOC 2" → "sock too"     "$299" → "two ninety nine" messy
#   "Professional plan" → "professional planned"
#
# Your earpiece Correction-GPT then starts from BAD text → worse coaching.
#
# Fine-tuning = show Whisper many (audio → correct transcript) examples from
# YOUR domain so those words become easy.
#
#   Generic Whisper          Domain-tuned Whisper
#   "sock too compliant"  →  "SOC 2 compliant"
#
# ---------------------------------------------------------------------------
# WHAT IS FINE-TUNING? (deep but easy)
# ---------------------------------------------------------------------------
# Fine-tuning = take a model that ALREADY learned a general skill, then keep
# training it a bit more on YOUR smaller, specialized dataset so it gets
# better at your job.
#
# Picture:
#   1) PRETRAIN (already done by OpenAI for Whisper)
#        Millions of hours of speech → model learns English sounds & spelling.
#        Like finishing school.
#
#   2) FINE-TUNE (what YOU do in this notebook)
#        Show pairs:  [sales audio] → "The Professional plan costs $299…"
#        Update weights (or LoRA adapters) so mistakes on sales words shrink.
#        Like an internship: same brain, new workplace vocabulary.
#
#   3) INFERENCE (your live agent)
#        Use the tuned model to transcribe new calls. No labels needed then.
#
# What changes under the hood?
#   Training still minimizes a loss (Whisper: predict next transcript tokens
#   given the audio). Difference vs "training from scratch":
#     - start from strong pretrained weights (not random)
#     - fewer steps / smaller LR usually
#     - dataset is domain-specific and much smaller
#
# Fine-tuning vs related ideas:
#   Transfer learning  — broad name for "reuse pretrained knowledge"
#   Fine-tuning        — the common transfer method: continue gradient updates
#                        on the pretrained net (full or LoRA)
#   Prompting / RAG    — change INPUTS, not weights (no training)
#   SFT (week2)        — fine-tuning a text LLM on instruction→answer flashcards
#                        (same IDEA as here; different modality: text vs speech)
#
# Sticky one-liner:
#   Fine-tuning = specialized practice for a pretrained model.
#
# ---------------------------------------------------------------------------
# FULL FINE-TUNE vs LoRA vs QLoRA (slow walkthrough)
# ---------------------------------------------------------------------------
# Think of Whisper as a HUGE binder of knobs (millions of weights).
# Fine-tuning means: turn some knobs so sales audio is recognized better.
#
# --- 1) FULL FINE-TUNE (Full FT) ---
#   What: unlock EVERY knob and train them all on your sales clips.
#   Pros: maximum flexibility; can change behavior a lot.
#   Cons:
#     - Needs a strong GPU / lots of memory
#     - With only 50–100 clips, the model can MEMORIZE those clips
#       and forget general English (catastrophic forgetting / overfit)
#     - Checkpoint is HUGE (you save the whole model again)
#
#   Picture: repainting the entire house to match one room's style.
#
# --- 2) LoRA (Low-Rank Adaptation) ---
#   What: FREEZE the original Whisper knobs (leave school knowledge intact).
#         Add tiny extra matrices (adapters) next to certain layers
#         (often attention projections). Train ONLY those tiny adapters.
#
#   Math intuition (no pain):
#     A big weight update would be a giant matrix ΔW.
#     LoRA says: approximate ΔW ≈ A × B where A,B are skinny/small ("low rank").
#     Far fewer numbers to learn → cheaper + less overfit on tiny data.
#
#   Pros:
#     - Train ~1% (or less) of parameters
#     - Small adapter file (MBs), base Whisper stays shared
#     - Swap adapters: sales_lora.pt vs medical_lora.pt without copying base
#   Cons:
#     - Slightly less flexible than full FT if you need a huge behavior change
#
#   Picture: leave the house painted; stick removable STYLE DECALS on doors.
#            Want a new domain? Peel decals, stick different ones.
#
#       ┌─────────────────────────────┐
#       │  Frozen Whisper weights     │  ← not updated
#       │    + LoRA adapters (train)  │  ← only these learn sales words
#       └─────────────────────────────┘
#
# --- 3) QLoRA (Quantized LoRA) ---
#   What: same LoRA idea, but the FROZEN base is stored in 4-bit (compressed
#         numbers) to save GPU RAM. Adapters still train in higher precision.
#
#   Quantization (simple):
#     Store weights with fewer bits (less detail) ≈ zip file for numbers.
#     Model is a bit "blurrier" but much smaller in memory.
#
#   Pros: fine-tune bigger Whisper variants on smaller GPUs / laptops
#   Cons: setup is pickier (bitsandbytes, GPU drivers); tiny quality tradeoff
#
#   Picture: keep the house as a compressed photo album (4-bit) + train
#            the same small decals (LoRA) on top.
#
# --- Which should YOU use for this sales Whisper project? ---
#   Tiny demo dataset (tens of clips) → prefer LoRA (or QLoRA if VRAM tight)
#   Full FT → only if you have lots of labeled audio + serious GPU
#
# Sticky cheat sheet:
#   Full FT = retrain whole brain
#   LoRA    = freeze brain, train small stickers
#   QLoRA   = freeze a COMPRESSED brain, train small stickers
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS
# ---------------------------------------------------------------------------
#   ASR  = Automatic Speech Recognition — speech → text (Whisper)
#   WER  = Word Error Rate — % of words wrong vs a gold transcript
#          (insertions + deletions + substitutions) / N_ref_words
#   LoRA = Low-Rank Adaptation — parameter-efficient fine-tuning method
#   TTS  = Text-To-Speech — can SYNTHESIZE training audio from scripts
#          (good for bootstrapping; real calls are better later)
#
# ---------------------------------------------------------------------------
# DATA YOU NEED
# ---------------------------------------------------------------------------
#   Pairs:  audio.wav  +  exact text that was said
#   Start:  50–100 sales lines (TTS or read aloud)
#   Better: real anonymized call snippets with human transcripts
#
# Pipeline in this notebook:
#   1) Build (audio, text) dataset
#   2) Load whisper-tiny.en + processor
#   3) (Later cells) LoRA train → evaluate WER → plug into live agent ASR
#
print("Whisper domain fine-tuning map loaded.")


Whisper domain fine-tuning map loaded.


In [27]:
# =============================================================================
# PREPARE SALES AUDIO + TRANSCRIPTS — (audio path, text) pairs for Whisper
# =============================================================================
#
# Goal of this cell:
#   Build a HuggingFace Dataset with columns like:
#     audio  → waveform @ 16 kHz (what Whisper hears)
#     text   → gold transcript (what Whisper should type)
#
# Production: real calls. Here: create a tiny sales_audio/ folder automatically
# if you don't have one yet (placeholder tones + real sales sentences).
# Replace with TTS (pyttsx3/Coqui) or mic recordings when you go serious.
#
# Terms:
#   WhisperProcessor — turns audio↔features and text↔token ids for Whisper
#   WhisperForConditionalGeneration — the seq2seq ASR model
#   sampling_rate 16000 — Whisper English checkpoints expect 16 kHz mono
#   metadata.csv — simple table: file name + transcript
#

import os
from pathlib import Path
import json
import numpy as np
import pandas as pd
import wave

from datasets import Dataset
from transformers import WhisperProcessor, WhisperForConditionalGeneration

AUDIO_DIR = Path("sales_audio")
META_CSV = AUDIO_DIR / "metadata.csv"
SAMPLE_RATE = 16000

# Domain lines you CARE about (pricing, compliance, plan names)
SALES_LINES = [
    ("call1.wav", "The Professional plan costs $299 per month."),
    ("call2.wav", "We are SOC 2 Type II compliant."),
    ("call3.wav", "Starter is $99 and includes 24/7 support."),
    ("call4.wav", "Enterprise customers get phone support with a one hour SLA."),
    ("call5.wav", "Rate limits are 5000 requests per minute for Professional."),
]


def _write_placeholder_wav(path: Path, seconds: float = 1.0, freq: float = 220.0):
    """Write a short mono 16-bit WAV (tone). Structure demo only — not real speech."""
    t = np.linspace(0, seconds, int(SAMPLE_RATE * seconds), endpoint=False)
    # Quiet tone so the file is valid audio; swap for TTS speech later
    audio = (0.1 * np.sin(2 * np.pi * freq * t) * 32767).astype(np.int16)
    with wave.open(str(path), "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(SAMPLE_RATE)
        wf.writeframes(audio.tobytes())


# Create folder + CSV + wavs if missing
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
if not META_CSV.exists():
    rows = []
    for i, (fname, text) in enumerate(SALES_LINES):
        wav_path = AUDIO_DIR / fname
        if not wav_path.exists():
            _write_placeholder_wav(wav_path, seconds=1.0 + 0.1 * i, freq=200 + 20 * i)
        rows.append({"file": fname, "text": text})
    pd.DataFrame(rows).to_csv(META_CSV, index=False)
    print(f"Created {META_CSV} with {len(rows)} placeholder clips.")
else:
    print(f"Using existing {META_CSV}")

df = pd.read_csv(META_CSV)


def _load_wav(path: Path) -> dict:
    """Load mono float32 waveform with stdlib wave — no torchcodec needed.

    datasets>=4 Audio() decoding requires the torchcodec package and a
    path→Audio cast that can break with pandas large_string columns.
    We already write 16 kHz WAVs above, so decode them ourselves.
    """
    with wave.open(str(path), "rb") as wf:
        assert wf.getframerate() == SAMPLE_RATE, f"expected {SAMPLE_RATE} Hz, got {wf.getframerate()}"
        n_ch = wf.getnchannels()
        raw = wf.readframes(wf.getnframes())
        arr = np.frombuffer(raw, dtype=np.int16).astype(np.float32) / 32768.0
        if n_ch > 1:
            arr = arr.reshape(-1, n_ch).mean(axis=1)
    return {"array": arr, "sampling_rate": SAMPLE_RATE}


rows = []
for _, row in df.iterrows():
    rows.append(
        {
            "file": row["file"],
            "text": row["text"],
            "audio": _load_wav(AUDIO_DIR / row["file"]),
        }
    )

dataset = Dataset.from_list(rows)

print("Sample row keys:", dataset[0].keys())
print("text:", dataset[0]["text"])
print("audio sampling_rate:", dataset[0]["audio"]["sampling_rate"])
print("audio array length:", len(dataset[0]["audio"]["array"]))

# Base English Whisper (tiny = fast for learning; upgrade to small/medium later)
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")

print("Loaded openai/whisper-tiny.en | trainable params:",
      sum(p.numel() for p in model.parameters()))
print("Next: freeze base + attach LoRA, then train on these (audio, text) pairs.")
print("NOTE: placeholder WAVs teach the pipeline; for real WER gains use TTS/real speech.")


Using existing sales_audio/metadata.csv
Sample row keys: dict_keys(['file', 'text', 'audio'])
text: The Professional plan costs $299 per month.
audio sampling_rate: 16000
audio array length: 16000


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 14155.91it/s]


Loaded openai/whisper-tiny.en | trainable params: 37760256
Next: freeze base + attach LoRA, then train on these (audio, text) pairs.
NOTE: placeholder WAVs teach the pipeline; for real WER gains use TTS/real speech.


In [28]:
# =============================================================================
# PREPROCESS DATA — turn (raw audio, text) into what Whisper trains on
# =============================================================================
#
# Sticky one-liner:
#   Preprocessing = convert HUMAN-friendly data → MODEL-friendly tensors
#   BEFORE the training loop. Same idea as tokenization for LLMs, but for
#   speech you also convert waveforms into spectrogram features.
#
# ---------------------------------------------------------------------------
# WHERE THIS SITS IN THE PIPELINE
# ---------------------------------------------------------------------------
#   RAW WORLD                 PREPROCESS (this cell)              TRAIN
#   -----------               ----------------------              -----
#   call1.wav  ─────────┐
#   "Professional…"     ├──→  input_features (log-Mel)  ──┐
#                       │     labels (token ids)          ├──→ loss / LoRA
#   (from previous cell)┘                                 │
#
# If you skip preprocess and feed raw float samples + English strings into
# Whisper, the model cannot learn — it only speaks "tensor language".
#
# ---------------------------------------------------------------------------
# WHAT IS PREPROCESSING? (deep but easy)
# ---------------------------------------------------------------------------
# Three jobs, always:
#   1) CLEAN / NORMALIZE  — make inputs comparable (rate, mono, scale, trim)
#   2) FEATURIZE          — turn signal into the representation the model expects
#   3) ALIGN LABELS       — encode targets the same way the model will decode
#
# For Whisper ASR specifically:
#   audio  → log-Mel spectrogram  →  input_features  shape (80, 3000)
#   text   → BPE token ids        →  labels          list of ints
#
# Picture (audio path):
#   waveform samples
#        │  STFT (short windows of sound)
#        ▼
#   frequency × time energy map
#        │  mel filterbank (human-ear frequency spacing)
#        ▼
#   80 mel bands × time frames
#        │  log + Whisper's per-feature normalize
#        ▼
#   input_features  ← THIS is what the encoder "hears"
#
# Picture (text path):
#   "The Professional plan costs $299 per month."
#        │  WhisperTokenizer (BPE + special tokens)
#        ▼
#   [50257, 50362, 464, 18612, ...]  ← decoder learning targets
#
# ---------------------------------------------------------------------------
# TYPES OF PREPROCESSING (map of the industry)
# ---------------------------------------------------------------------------
# A) DATA-QUALITY / CLEANING (often offline, before Dataset)
#    - silence trim, denoise, gain normalize, drop corrupt files
#    - language / PII filtering, transcript QA
#    Industry: common in ASR data pipelines (Kaldi / NVIDIA NeMo / SpeechBrain)
#
# B) SIGNAL / ACOUSTIC NORMALIZATION
#    - resample to model rate (Whisper: 16 kHz mono)
#    - peak / loudness normalize (e.g. target LUFS in production podcasts)
#    - channel mix (stereo → mono)
#    Industry standard for English Whisper: 16_000 Hz, mono, float in [-1, 1]
#
# C) FEATURE EXTRACTION (what THIS cell focuses on for audio)
#    Classic ASR eras used different "ears":
#      MFCC          — Mel-Frequency Cepstral Coefficients (GMM-HMM / early DNN)
#      Filterbank / Mel spectrogram — most modern neural ASR
#      Raw waveform  — some end-to-end models (Wav2Vec 2.0 learns its own front-end)
#      log-Mel       — Whisper / many Transformer ASR models (OpenAI standard)
#    Whisper fixed recipe (industry de-facto for this family):
#      - 25 ms window (n_fft=400 @ 16 kHz), hop 10 ms (160 samples)
#      - 80 mel bins
#      - pad / truncate to 30 seconds → 3000 time frames
#      → tensor shape (80, 3000) per clip
#
# D) TEXT / LABEL PREPROCESSING
#    - normalize transcripts (numbers, casing, punctuation policy)
#    - tokenize with the SAME tokenizer the checkpoint was trained with
#    - special tokens: Whisper uses start/transcript markers around the text
#    Industry: NEVER mix tokenizers across models; mismatch = silent garbage.
#
# E) AUGMENTATION (train-time preprocess; optional later cell)
#    - SpecAugment (mask time/freq on the Mel), speed perturb, noise mix
#    Industry: SpecAugment is a standard for Transformer ASR; helps tiny datasets.
#
# F) BATCHING / COLLATION (DataLoader preprocess)
#    - pad variable-length labels to a batch max
#    - mask padding with -100 so CrossEntropy ignores it (HF Trainer convention)
#    Not in this cell yet — comes when you wire Trainer / custom collator.
#
# G) LLM-SIDE PREPROCESS (for comparison — week2 SFT)
#    - chat template + BPE tokenize + label mask on prompt tokens
#    Same IDEA as here: strings → ids. Different MODALITY on the input side.
#
# ---------------------------------------------------------------------------
# INDUSTRY STANDARDS (what "good" looks like)
# ---------------------------------------------------------------------------
# Speech ASR:
#   ✓ Match the pretrained model's front-end exactly (rate, mel, hop, pad)
#   ✓ Train / eval transcript normalization policy must match how you score WER
#   ✓ Prefer storing paths + decode on the fly for big corpora; tiny demo = OK
#      to materialize features in RAM (what we do below)
#   ✓ Document special-token handling (Whisper: decoder prompt tokens in labels)
#   ✗ Don't invent your own Mel and expect a pretrained Whisper to transfer
#
# Vision / multimodal (same philosophy elsewhere):
#   ImageNet-style mean/std, fixed resize/crop — always match the checkpoint.
#
# Text LLMs:
#   Chat template + tokenizer from THAT model card — same rule as WhisperTokenizer.
#
# Rule of thumb used in labs & production:
#   "Preprocess like the original training run, or fine-tuning is fighting you."
#
# ---------------------------------------------------------------------------
# ABBREVIATIONS
# ---------------------------------------------------------------------------
#   STFT     = Short-Time Fourier Transform — windowed frequency analysis
#   Mel      = Mel scale — frequency axis warped like human hearing
#   log-Mel  = log of mel filterbank energies (Whisper input)
#   BPE      = Byte-Pair Encoding — tokenizer Whisper uses for text
#   WER      = Word Error Rate — how we judge ASR quality later
#   LUFS     = loudness units (broadcast / podcast loudness standard)
#   SpecAug  = SpecAugment — mask patches on spectrogram as augmentation
#
# ---------------------------------------------------------------------------
# THIS CELL'S CODE (what each line does)
# ---------------------------------------------------------------------------
#   processor = WhisperProcessor...
#       Loads BOTH:
#         feature_extractor  → audio → input_features
#         tokenizer          → text  → labels
#       Always load the SAME checkpoint name as the model ("tiny.en").
#
#   prepare_dataset(batch):
#       audio["array"]  → feature_extractor → batch["input_features"]
#       batch["text"]   → tokenizer         → batch["labels"]
#
#   dataset.map(..., remove_columns=...)
#       Drops raw audio/text columns; keeps only tensors the trainer needs.
#
#   set_format("torch", ...)
#       So next cells get torch.Tensors instead of Python lists.
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT (shapes to remember)
# ---------------------------------------------------------------------------
#   input_features[0]  → shape (80, 3000)
#       80  = mel bins (frequency "piano keys")
#       3000 = time frames for a 30s Whisper chunk (short clips are zero-padded)
#   labels[0]          → e.g. ~10–40 ints for our short sales sentences
#       Starts with Whisper special ids, then BPE pieces of the transcript
#   After decode(labels, skip_special_tokens=True) → original English text
#

# Reuse processor if the previous cell already loaded it; otherwise load here.
try:
    processor
except NameError:
    processor = WhisperProcessor.from_pretrained("openai/whisper-tiny.en")


def prepare_dataset(batch):
    """One example → Whisper training tensors.

    Input batch (from previous cell):
      batch["audio"]["array"]          float32 waveform @ 16 kHz
      batch["audio"]["sampling_rate"]  16000
      batch["text"]                    gold transcript string

    Output batch:
      batch["input_features"]  log-Mel, length-80×3000 (list/np before set_format)
      batch["labels"]          token ids the decoder should emit
    """
    audio = batch["audio"]

    # --- AUDIO → log-Mel features (encoder input) ---
    # WhisperFeatureExtractor:
    #   1) optional resample (we already are at 16 kHz)
    #   2) STFT → mel filterbank → log
    #   3) pad/trim to 30s → (80, 3000)
    feats = processor.feature_extractor(
        audio["array"],
        sampling_rate=16000,  # MUST match Whisper English checkpoints
    )
    batch["input_features"] = feats.input_features[0]

    # --- TEXT → token ids (decoder targets) ---
    # Same tokenizer Whisper was pretrained with (BPE + special tokens).
    batch["labels"] = processor.tokenizer(batch["text"]).input_ids
    return batch


# Map over the whole tiny sales set (5 clips). For 10k+ hours you'd:
#   - decode audio lazily, cache features on disk, or use streaming datasets.
dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)
dataset.set_format(type="torch", columns=["input_features", "labels"])

# ----- walk the OUTPUT so you can "see" preprocess -----
ex = dataset[0]
feat = ex["input_features"]
labs = ex["labels"]

print("columns after preprocess:", dataset.column_names)
print("input_features shape:", tuple(feat.shape), " dtype:", feat.dtype)
print("  → 80 mel bins × 3000 time frames (Whisper 30s canvas)")
print("labels (token ids):", labs.tolist() if hasattr(labs, "tolist") else labs)
print("labels length:", len(labs))
print(
    "labels → text:",
    processor.tokenizer.decode(labs, skip_special_tokens=True),
)
print("Ready for training: each row is (encoder Mel, decoder token targets).")


Map: 100%|██████████| 5/5 [00:00<00:00, 317.73 examples/s]

columns after preprocess: ['input_features', 'labels']
input_features shape: (80, 3000)  dtype: torch.float32
  → 80 mel bins × 3000 time frames (Whisper 30s canvas)
labels (token ids): [50257, 50362, 464, 18612, 1410, 3484, 720, 22579, 583, 1227, 13, 50256]
labels length: 12
labels → text: The Professional plan costs $299 per month.
Ready for training: each row is (encoder Mel, decoder token targets).


In [29]:
# =============================================================================
# APPLY LoRA TO WHISPER — freeze the brain, train tiny "stickers"
# =============================================================================
#
# Sticky one-liner:
#   LoRA = leave pretrained Whisper weights FROZEN, add small adapter
#   matrices next to attention projections, train ONLY those adapters.
#
# ---------------------------------------------------------------------------
# WHY NOT FULL FINE-TUNE HERE?
# ---------------------------------------------------------------------------
# whisper-tiny.en ≈ 37.8M params. Full FT updates all of them.
# You have ~5 demo clips. Updating 37M knobs on 5 examples → memorizes
# tones, forgets English (catastrophic forgetting).
#
# LoRA updates ~0.4% of params (q_proj / v_proj adapters only).
# Same goal (sales jargon) with far less overfit + a tiny save file.
#
# Picture:
#   Frozen Whisper W          +   LoRA  ΔW ≈ A @ B   (skinny × skinny)
#   (school knowledge)            (sales stickers you train)
#
# ---------------------------------------------------------------------------
# WHAT EACH HYPERPARAMETER MEANS
# ---------------------------------------------------------------------------
#   r (rank)        — width of the skinny adapters. 8 = small/cheap.
#                     Bigger r → more capacity, more VRAM, more overfit risk.
#   lora_alpha      — scales the adapter update (alpha/r). 16 with r=8 → scale 2.
#   target_modules  — which linear layers get stickers.
#                     Whisper attention uses q_proj / k_proj / v_proj / out_proj.
#                     Industry default start: q_proj + v_proj (Hu et al. LoRA paper).
#   lora_dropout    — dropout on adapters (regularization). 0.05–0.1 typical.
#   bias            — "none" = don't train bias terms (usual).
#   task_type       — OMIT for Whisper. SEQ_2_SEQ_LM wraps PeftModelForSeq2SeqLM
#                     (T5-style), which injects input_ids into WhisperDecoder and
#                     crashes: "multiple values for keyword argument input_ids".
#
# ---------------------------------------------------------------------------
# THIS CELL'S CODE
# ---------------------------------------------------------------------------
#   LoraConfig(...)     → recipe for adapters
#   get_peft_model(...) → wrap Whisper, freeze base, inject LoRA
#   print_trainable_parameters() → sanity check: should be << 1%
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT (your run)
# ---------------------------------------------------------------------------
#   trainable params: 147,456 || all params: 37,907,712 || trainable%: 0.3890
#   Meaning: ~147k numbers learn sales words; ~37.8M stay frozen.
#

from peft import LoraConfig, get_peft_model
from transformers import WhisperForConditionalGeneration

# Fresh base (don't LoRA-wrap an already-wrapped model if you re-run this cell)
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")
# Training: disable KV cache (it's an inference speed trick; fights gradients)
model.config.use_cache = False

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Whisper attention projections
    lora_dropout=0.1,
    bias="none",
    # no task_type: Whisper is audio seq2seq, not T5 text seq2seq (see note above)
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("LoRA attached. Next cell: Trainer + collator (batch padding / -100 labels).")


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 13379.09it/s]


trainable params: 147,456 || all params: 37,907,712 || trainable%: 0.3890
LoRA attached. Next cell: Trainer + collator (batch padding / -100 labels).


In [32]:
# =============================================================================
# TRAINING SETUP — Seq2SeqTrainer + speech collator
# =============================================================================
#
# Sticky one-liner:
#   Trainer = the training loop (forward → loss → backward → step).
#   Collator = how several examples become ONE padded batch.
#
# Whisper is seq2seq: encoder sees Mel features, decoder predicts token ids.
# HF's Seq2SeqTrainer is the industry-standard loop for that family.
#
# ---------------------------------------------------------------------------
# FIXES vs older tutorials (transformers 5.x)
# ---------------------------------------------------------------------------
#   evaluation_strategy  →  eval_strategy     (renamed)
#   tokenizer=...        →  processing_class= (WhisperProcessor, not a tokenizer)
#   warmup_steps=50 on a 5-clip set would keep LR near 0 the whole time → 0 here
#   remove_unused_columns=False  — PEFT wraps forward(); Trainer would otherwise
#                                  DROP input_features thinking they're unused
#
# ---------------------------------------------------------------------------
# TRAINING ARGUMENTS (what each knob does)
# ---------------------------------------------------------------------------
#   output_dir              checkpoints + logs
#   per_device_train_batch_size  clips per GPU/CPU step (2 = tiny-demo friendly)
#   gradient_accumulation_steps  simulate a bigger batch by summing N tiny steps
#                                (production: 4–16; here 1 because only 5 clips)
#   learning_rate           LoRA often likes 1e-4 .. 5e-4 (higher than full FT)
#   warmup_steps            linearly ramp LR; use when you have MANY steps
#   num_train_epochs        full passes over the dataset
#   logging_steps           print loss every N optimizer steps
#   eval_strategy "no"      skip eval loop (no held-out set yet)
#   save_strategy "no"      skip Trainer checkpoints (see tied-weight note)
#   predict_with_generate   eval via model.generate (not teacher-forced logits)
#   generation_max_length   cap decode length during generate-eval
#   report_to "none"        don't spam wandb/tensorboard
#
# Dependency: accelerate (>=1.1.0) — HF Trainer's device/multi-GPU backend.
#   pip install 'accelerate>=1.1.0'   (must be the SAME python as this kernel)
#
# Effective batch = per_device_train_batch_size × gradient_accumulation_steps
#                 × num_gpus
#
# ---------------------------------------------------------------------------
# DATA COLLATOR (why we need one)
# ---------------------------------------------------------------------------
# Rows have:
#   input_features  — already fixed (80, 3000)  → pad is basically a stack
#   labels          — VARIABLE length token ids → pad to batch max length
#
# Padding labels with tokenizer pad_id would make the model LEARN to predict
# pad tokens. Industry standard: replace pad with -100 so CrossEntropyLoss
# ignores those positions (PyTorch ignore_index=-100).
#
# Picture of one batch (batch_size=2):
#   input_features:  (2, 80, 3000)
#   labels:          (2, L_max)   with -100 in the padded tail
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT
# ---------------------------------------------------------------------------
#   Trainer ready. train rows: 5  columns: ['input_features', 'labels']
#   (No crash on Seq2SeqTrainingArguments. trainer is now defined.)
#
# Whisper ties decoder.embed_tokens ↔ proj_out (same storage). If Trainer
# saves a FULL state_dict via safetensors it crashes:
#   RuntimeError: tensors share memory: proj_out.weight / embed_tokens.weight
# PEFT's model.save_pretrained() only writes tiny adapter files — do that
# after train(), not via Trainer epoch checkpoints.
#

from dataclasses import dataclass
from typing import Any, Dict, List, Union
import importlib
import subprocess
import sys

import torch


def _ensure_hf_accelerate():
    """HF Trainer needs `accelerate` in THIS notebook kernel.

    Two traps:
      1) Jupyter kernel != terminal pip target → install via sys.executable
      2) transformers caches is_accelerate_available() at import time.
         If you imported transformers before installing accelerate, the cache
         stays False until we clear it and reload training_args/trainer.
    """
    try:
        import accelerate  # noqa: F401
    except ImportError:
        print(f"Installing accelerate into kernel: {sys.executable}")
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", "-q", "accelerate>=1.1.0"]
        )
        import accelerate  # noqa: F401

    import transformers.utils.import_utils as iu
    for name in ("is_accelerate_available", "_is_package_available"):
        fn = getattr(iu, name, None)
        if fn is not None and hasattr(fn, "cache_clear"):
            fn.cache_clear()

    import transformers.training_args as ta
    import transformers.training_args_seq2seq as tas
    importlib.reload(ta)
    importlib.reload(tas)
    # Do NOT reload Trainer: that recreates PeftModel's class identity and
    # Trainer then dumps a FULL Whisper state_dict → safetensors tied-weight crash.
    from transformers import Seq2SeqTrainer
    print("accelerate", accelerate.__version__, "| Trainer backend ready")
    return tas.Seq2SeqTrainingArguments, Seq2SeqTrainer


Seq2SeqTrainingArguments, Seq2SeqTrainer = _ensure_hf_accelerate()

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-sales",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=1,  # tiny dataset: 1 step ≈ 1 real update
    learning_rate=1e-4,
    warmup_steps=0,                 # 5 clips → few steps; warmup would starve LR
    num_train_epochs=3,
    logging_steps=1,                # see every step on a tiny run
    eval_strategy="no",             # transformers 5.x name (not evaluation_strategy)
    save_strategy="no",             # avoid Trainer safetensors + Whisper tied weights
    predict_with_generate=True,
    generation_max_length=225,
    report_to="none",
    remove_unused_columns=False,    # required with PEFT + Whisper features
    label_names=["labels"],
    fp16=False,                     # safer on Mac CPU/MPS
    dataloader_pin_memory=False,
)


@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    """Pad Mel features + label ids; mask label pads with -100."""

    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Audio features and text labels have different shapes → pad separately
        input_features = [{"input_features": f["input_features"]} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": f["labels"]} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # ignore_index for CrossEntropyLoss
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)
        # ONLY these two keys. Extra masks/input_ids confuse Whisper + PEFT.
        return {"input_features": batch["input_features"], "labels": labels}


data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator,
    # no processing_class: collator already returns tensors. Passing WhisperProcessor
    # can make Trainer inject text input_ids (wrong modality for Whisper).
)

print("Trainer ready. train rows:", len(dataset), " columns:", dataset.column_names)
print("Next: trainer.train() then save LoRA adapters.")


accelerate 1.14.0 | Trainer backend ready
Trainer ready. train rows: 5  columns: ['input_features', 'labels']
Next: trainer.train() then save LoRA adapters.


In [33]:
# =============================================================================
# TRAIN AND SAVE — run the loop, keep only LoRA adapters + processor
# =============================================================================
#
# Sticky one-liner:
#   trainer.train() walks batches → loss → update LoRA A/B matrices.
#   save_pretrained on a PEFT model writes ADAPTERS only (MBs), not full Whisper.
#
# ---------------------------------------------------------------------------
# WHAT YOU SHOULD SEE
# ---------------------------------------------------------------------------
#   A short HF progress bar with loss numbers each logging_steps.
#   loss starting somewhere (often ~ few nats) and hopefully trending down.
#
#   With PLACEHOLDER TONES (not real speech) loss may not become "good ASR".
#   This cell proves the PIPELINE. Swap in TTS / real calls for real WER.
#
# ---------------------------------------------------------------------------
# WHAT GETS SAVED
# ---------------------------------------------------------------------------
#   Trainer checkpoints (./whisper-sales/checkpoint-*) are OFF on purpose.
#   Whisper ties embed_tokens ↔ lm_head/proj_out; safetensors refuses that
#   shared memory. We save adapters ourselves after train() instead:
#
#   ./whisper-sales-lora/
#     adapter_config.json      — LoRA recipe (r, alpha, target_modules, ...)
#     adapter_model.safetensors (or .bin) — trained sticker weights
#     preprocessor / tokenizer files from processor.save_pretrained
#
#   Base whisper-tiny.en is NOT copied. Reload later =
#     WhisperForConditionalGeneration(base) + PeftModel.from_pretrained(adapters)
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT (shape, not exact loss)
# ---------------------------------------------------------------------------
#   {'train_runtime': ..., 'train_loss': <float>, 'epoch': 3.0}
#   Saved LoRA adapters + processor → ./whisper-sales-lora
#

train_result = trainer.train()
print("train metrics:", dict(train_result.metrics))

SAVE_DIR = "./whisper-sales-lora"
model.save_pretrained(SAVE_DIR)       # PEFT → adapters only
processor.save_pretrained(SAVE_DIR)   # feature extractor + tokenizer
print(f"Saved LoRA adapters + processor → {SAVE_DIR}")
print("Reload recipe: base whisper-tiny.en + PeftModel.from_pretrained(SAVE_DIR)")


Step,Training Loss
1,7.127704
2,7.093617
3,7.502577
4,7.189840
5,6.990127
6,7.014034
7,6.948283
8,7.106409
9,6.972355


train metrics: {'train_runtime': 1.4187, 'train_samples_per_second': 10.573, 'train_steps_per_second': 6.344, 'total_flos': 372468326400000.0, 'train_loss': 7.104994085099962, 'epoch': 3.0}
Saved LoRA adapters + processor → ./whisper-sales-lora
Reload recipe: base whisper-tiny.en + PeftModel.from_pretrained(SAVE_DIR)


In [34]:
# =============================================================================
# EVALUATE WER — how wrong is the transcript vs gold text?
# =============================================================================
#
# Sticky one-liner:
#   WER = Word Error Rate = (substitutions + deletions + insertions) / N_ref_words
#   Lower is better. 0.0 = every word matches. 1.0 = as many edits as ref words.
#
# ---------------------------------------------------------------------------
# WHY NOT `import evaluate`?
# ---------------------------------------------------------------------------
# HuggingFace `evaluate` + `jiwer` are industry-common, but extra deps.
# WER is just token-level Levenshtein distance / len(reference words).
# We compute it here so the notebook runs without that package.
#
# Industry note: production still often uses jiwer/evaluate, AND a
# normalization policy (lowercase, expand numbers, strip punctuation) that
# MUST match how you trained transcripts — otherwise WER lies.
#
# ---------------------------------------------------------------------------
# THIS CELL'S CODE
# ---------------------------------------------------------------------------
#   1) Reload sales_audio wavs (preprocess cell dropped raw audio columns)
#   2) model.generate → predicted text
#   3) WER(pred, gold) per clip + micro-average
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT
# ---------------------------------------------------------------------------
#   One line per clip: gold | pred | clip WER
#   Then: overall WER: <float>
#
#   HONEST DEMO NOTE:
#   Placeholder sine-wave WAVs are not speech. WER will look BAD even after
#   "training". The metric + generate path is what you're learning. For a
#   real drop (jargon 25% → <5%) you need TTS or real spoken sales lines.
#

from pathlib import Path

import numpy as np
import pandas as pd
import torch
import wave


def _edit_distance(a, b):
    """Levenshtein distance between two token lists (insert/delete/substitute)."""
    n, m = len(a), len(b)
    dp = list(range(m + 1))
    for i in range(1, n + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, m + 1):
            cur = dp[j]
            if a[i - 1] == b[j - 1]:
                dp[j] = prev
            else:
                dp[j] = 1 + min(prev, dp[j], dp[j - 1])  # sub, del, ins
            prev = cur
    return dp[m]


def word_error_rate(predictions, references):
    """Micro-averaged WER. Industry formula: edits / reference word count."""
    edits, n_ref = 0, 0
    for pred, ref in zip(predictions, references):
        p_tok = pred.lower().split()
        r_tok = ref.lower().split()
        edits += _edit_distance(p_tok, r_tok)
        n_ref += max(len(r_tok), 1)
    return edits / n_ref


def _load_wav_mono_16k(path: Path):
    with wave.open(str(path), "rb") as wf:
        sr = wf.getframerate()
        n_ch = wf.getnchannels()
        arr = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
        if n_ch > 1:
            arr = arr.reshape(-1, n_ch).mean(axis=1)
    if sr != 16000:
        dur = len(arr) / float(sr)
        t_old = np.linspace(0.0, dur, num=len(arr), endpoint=False)
        t_new = np.linspace(0.0, dur, num=int(dur * 16000), endpoint=False)
        arr = np.interp(t_new, t_old, arr).astype(np.float32)
        sr = 16000
    return arr, sr


def evaluate_on_test(test_samples, gen_model, gen_processor):
    device = next(gen_model.parameters()).device
    predictions, references = [], []
    gen_model.eval()
    for sample in test_samples:
        arr, sr = sample["array"], sample["sampling_rate"]
        feats = gen_processor(arr, sampling_rate=sr, return_tensors="pt").input_features.to(device)
        with torch.no_grad():
            ids = gen_model.generate(feats, max_new_tokens=64)
        pred = gen_processor.batch_decode(ids, skip_special_tokens=True)[0].strip()
        gold = sample["text"]
        clip_wer = word_error_rate([pred], [gold])
        print(f"  gold: {gold!r}")
        print(f"  pred: {pred!r}")
        print(f"  clip WER: {clip_wer:.3f}\n")
        predictions.append(pred)
        references.append(gold)
    wer = word_error_rate(predictions, references)
    print(f"overall WER: {wer:.4f}")
    return wer


AUDIO_DIR = Path("sales_audio")
meta = pd.read_csv(AUDIO_DIR / "metadata.csv")
test_data = []
for row in meta.itertuples(index=False):
    arr, sr = _load_wav_mono_16k(AUDIO_DIR / row.file)
    test_data.append({"file": row.file, "text": row.text, "array": arr, "sampling_rate": sr})

print(f"Evaluating {len(test_data)} clips from {AUDIO_DIR}/ (placeholder tones → high WER is expected)\n")
evaluate_on_test(test_data, model, processor)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Both `max_new_tokens` (=64) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_p

Evaluating 5 clips from sales_audio/ (placeholder tones → high WER is expected)

  gold: 'The Professional plan costs $299 per month.'
  pred: 'you'
  clip WER: 1.000

  gold: 'We are SOC 2 Type II compliant.'
  pred: 'you'
  clip WER: 1.000

  gold: 'Starter is $99 and includes 24/7 support.'
  pred: 'you'
  clip WER: 1.000



[transformers] Both `max_new_tokens` (=64) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


  gold: 'Enterprise customers get phone support with a one hour SLA.'
  pred: 'you'
  clip WER: 1.000

  gold: 'Rate limits are 5000 requests per minute for Professional.'
  pred: 'You'
  clip WER: 1.000

overall WER: 1.0000


1.0

In [35]:
# =============================================================================
# INTEGRATE INTO THE LIVE ASSISTANT — drop-in ASR class
# =============================================================================
#
# Sticky one-liner:
#   FineTunedASR = load base Whisper + your LoRA adapters, transcribe 16 kHz audio.
#   Swap this in for the ASR step in week2 live earpiece loop.
#
# ---------------------------------------------------------------------------
# HOW RELOAD WORKS (easy to get wrong)
# ---------------------------------------------------------------------------
#   model.save_pretrained("./whisper-sales-lora") saved ADAPTERS, not a full
#   Whisper checkpoint. This would FAIL:
#     WhisperForConditionalGeneration.from_pretrained("./whisper-sales-lora")
#
#   Correct:
#     base = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny.en")
#     model = PeftModel.from_pretrained(base, "./whisper-sales-lora")
#
# Picture:
#   [frozen tiny.en from Hub] + [sales LoRA folder] → ASR that knows your jargon
#
# ---------------------------------------------------------------------------
# THIS CELL'S CODE
# ---------------------------------------------------------------------------
#   FineTunedASR.__init__   load processor + PeftModel, eval mode
#   transcribe_file(path)   wave → Mel → generate → text  (no torchaudio needed)
#   transcribe(audio_bytes) same, from in-memory WAV bytes (live-mic path)
#
# ---------------------------------------------------------------------------
# EXPECTED OUTPUT
# ---------------------------------------------------------------------------
#   Loaded LoRA ASR from ./whisper-sales-lora on device: cpu|mps|cuda
#   demo file: sales_audio/call2.wav
#   transcript: '...'   (garbage-ish on sine tones; real speech later)
#

from pathlib import Path
import os
import tempfile

import numpy as np
import torch
import wave
from peft import PeftModel
from transformers import WhisperForConditionalGeneration, WhisperProcessor


class FineTunedASR:
    """Drop-in ASR: base Whisper + saved LoRA adapters."""

    def __init__(self, model_path="./whisper-sales-lora", base_model="openai/whisper-tiny.en"):
        self.processor = WhisperProcessor.from_pretrained(model_path)
        base = WhisperForConditionalGeneration.from_pretrained(base_model)
        self.model = PeftModel.from_pretrained(base, model_path)
        self.model.eval()
        self.device = next(self.model.parameters()).device
        self.model.to(self.device)

    def _wav_to_array(self, path):
        with wave.open(str(path), "rb") as wf:
            sr = wf.getframerate()
            n_ch = wf.getnchannels()
            arr = np.frombuffer(wf.readframes(wf.getnframes()), dtype=np.int16).astype(np.float32) / 32768.0
            if n_ch > 1:
                arr = arr.reshape(-1, n_ch).mean(axis=1)
        if sr != 16000:
            dur = len(arr) / float(sr)
            t_old = np.linspace(0.0, dur, num=len(arr), endpoint=False)
            t_new = np.linspace(0.0, dur, num=int(dur * 16000), endpoint=False)
            arr = np.interp(t_new, t_old, arr).astype(np.float32)
            sr = 16000
        return arr, sr

    def transcribe_file(self, path):
        arr, sr = self._wav_to_array(path)
        feats = self.processor(arr, sampling_rate=sr, return_tensors="pt").input_features.to(self.device)
        with torch.no_grad():
            ids = self.model.generate(feats, max_new_tokens=64)
        return self.processor.batch_decode(ids, skip_special_tokens=True)[0].strip()

    def transcribe(self, audio_bytes: bytes):
        """Live-agent path: WAV bytes from the mic buffer."""
        with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as tmp:
            tmp.write(audio_bytes)
            tmp_path = tmp.name
        try:
            return self.transcribe_file(tmp_path)
        finally:
            os.unlink(tmp_path)


SAVE_DIR = "./whisper-sales-lora"
demo_wav = Path("sales_audio") / "call2.wav"

asr = FineTunedASR(SAVE_DIR)
print(f"Loaded LoRA ASR from {SAVE_DIR} on device: {asr.device}")
print("demo file:", demo_wav)
print("transcript:", repr(asr.transcribe_file(demo_wav)))
print("Plug FineTunedASR.transcribe(audio_bytes) into the live earpiece ASR step.")


Loading weights: 100%|██████████| 167/167 [00:00<00:00, 7475.36it/s]
[transformers] Both `max_new_tokens` (=64) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Loaded LoRA ASR from ./whisper-sales-lora on device: cpu
demo file: sales_audio/call2.wav
transcript: 'you'
Plug FineTunedASR.transcribe(audio_bytes) into the live earpiece ASR step.
